# Extracting Features from Text Data 

## Reading the Review Text from Our Feature Table

In [0]:
%python
text_df = spark.table("pipeline_catalog.tabular_pipeline.customer_features_final").select("customer_id", "review_text")
print(f"Total reviews: {text_df.count()}")
text_df.show(5, truncate=False)

## Cleaning the Text for Word-Level Analysis
Cleaning steps in this code

- lower → all lowercase → “Great” = “great”
- trim → removes leading and trailing spaces
- regexp_replace → removes punctuation, emojis and special characters → keeps letters, digits and spaces
- Result → new column cleaned_review

In [0]:
%python
from pyspark.sql.functions import lower, trim, regexp_replace, col

cleaned_df = text_df.withColumn(
    "cleaned_review",
    regexp_replace(trim(lower(col("review_text"))), r"[^a-z0-9\s]", "")
)

cleaned_df.select("customer_id", "review_text", "cleaned_review").show(5, truncate=False)

## Calculating Token Counts
- Splits each cleaned review into words (`tokens`) and counts them (`token_count`) to get the review length as a feature.
- A 2-word review carries far fewer signals than a 20-word review.

In [0]:
%python
from pyspark.sql.functions import split, size

token_counts_df = cleaned_df \
    .withColumn("tokens", split(col("cleaned_review"), r"\s+")) \
    .withColumn("token_count", size(col("tokens")))

token_counts_df.select("customer_id", "cleaned_review", "token_count").show(10, truncate=False)

## Calculating Sentence Length
- Splits the original review on `.`, `!`, `?` to count sentences (`sentence_count`), then divides `token_count` by it to get `avg_sentence_length`.
- Uses `review_text` instead of `cleaned_review` because cleaning removed the punctuation needed to detect sentences.


In [0]:
%python
from pyspark.sql.functions import array_remove, when, round as _round

sentence_df = token_counts_df \
    .withColumn("sentence_count", size(array_remove(split(col("review_text"), r"[.!?]+"), ""))) \
    .withColumn(
        "avg_sentence_length",
        when(col("sentence_count") == 0, 0)
        .otherwise(_round(col("token_count") / col("sentence_count"), 2))
    )

sentence_df.select("customer_id", "review_text", "sentence_count", "token_count", "avg_sentence_length").show(10, truncate=False)

# TF-IDF
- Converts each review into numbers a model can use, giving every word an importance score. Common words score low. Rare/distinctive scores high
- Words present in almost every review (`food`) get a low score; words present in few reviews (`spilled`, `cold`) get a high score.
- The model focuses on words that distinguish one review from another.
- Limitation: no understanding of meaning (`good` ≠ `great`), which is why embeddings are used next.

## Tokenizing and Removing Stop Words for TF-IDF

In [0]:
# To check the Pyspark's list of stopwords
from pyspark.ml.feature import Tokenizer, StopWordsRemover
StopWordsRemover.loadDefaultStopWords("english")

In [0]:
tokenizer = Tokenizer(inputCol="cleaned_review", outputCol="words")
tokenized_df = tokenizer.transform(sentence_df)

remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
filtered_df = remover.transform(tokenized_df)

filtered_df.select("customer_id", "words", "filtered_words").show(5, truncate=False)

## Computing Term Frequency
- `CountVectorizer` builds a vocabulary of words and counts each word per review → `raw_tf` (sparse vector).
- `minDF=2` → filter: drops words found in fewer than 2 reviews. TF-IDF rewards rare words, but words that are *too* rare (typos, names) are noise the model can't learn from → avoids overfitting.
- `vocabSize=200` → cap: at most 200 words (columns), even if more pass the filter.
- `99` → number of words that passed the filter (below the 200 cap).
- `[:15]` → only a preview of the vocabulary for display.

In [0]:
%python
from pyspark.ml.feature import CountVectorizer

cv = CountVectorizer(inputCol="filtered_words", outputCol="raw_tf", vocabSize=200, minDF=2.0)
cv_model = cv.fit(filtered_df)
tf_df = cv_model.transform(filtered_df)

print("Vocabulary sample:", cv_model.vocabulary[:15])

tf_df.select("customer_id", "filtered_words", "raw_tf").show(5, truncate=False)

The vector doesn’t keep the word order of the sentence. This is the bag-of-words principle: we know which words are present and how many times, but not in which order.

In [0]:
print((cv_model.vocabulary)) 

We see that :
- food is everywhere → it doesn’t distinguish any review → TF-IDF gives it a low weight.
- spilled is rare → it signals a specific issue → TF-IDF gives it a high weight.


## Computing TF-IDF

## Why IDF is needed
- In review 1004, `food` and `fresh` each appear once → TF gives both the same score (1.0)
- But `food` is in almost every review, `fresh` in far fewer → TF ignores this
- TF only looks at the current review → IDF looks at all reviews to correct the score
- Result → `food` ≈ 0.45 (common), `fresh` ≈ 2.83 (distinctive)


$$IDF(word) = \log\left(\frac{N + 1}{DF(word) + 1}\right)$$

$$TF\text{-}IDF = TF \times IDF$$

- `N` → total number of reviews
- `DF(word)` → number of reviews containing the word
- `+1` → smoothing: avoids division by zero; a word in every review gets `log(1) = 0`
- `log` → compresses the scale so rare words weigh more without dominating
- `idf.fit()` computes the IDF weight of each word → `transform()` multiplies each TF count by it → `tfidf_features`

In [0]:
%python
from pyspark.ml.feature import IDF

idf = IDF(inputCol="raw_tf", outputCol="tfidf_features")
idf_model = idf.fit(tf_df)
tfidf_df = idf_model.transform(tf_df)

tfidf_df.select("customer_id", "tfidf_features").show(5, truncate=False)

## Converting TF-IDF Vectors into Storable Arrays

- `vector_to_array` turns the sparse ML `Vector` into a standard `array<double>` → readable in SQL and storable in Delta / feature tables.
- The array is dense: one value per vocabulary word (99), so absent words become `0.0`.

In [0]:
%python
from pyspark.ml.functions import vector_to_array

final_df = tfidf_df.withColumn("tfidf_array", vector_to_array(col("tfidf_features")))

final_df.select("customer_id", "token_count", "sentence_count", "avg_sentence_length", "tfidf_array").show(5, truncate=False)

## Writing the Text Features Table

In [0]:
%python
final_df.select(
    "customer_id", "token_count", "sentence_count", "avg_sentence_length", "tfidf_array"
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pipeline_catalog.tabular_pipeline.customer_text_features")

print("customer_text_features Delta table created successfully")